In [7]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

In [8]:
#!pip install -q youtube-transcript-api langchain-groq langchain-community faiss-cpu python-dotenv youtube-transcript-api tiktoken



In [9]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

c:\Users\normk\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\normk\AppData\Local\Temp\ipykernel_10576\3936181027.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [10]:
video_id = "LPZh9BOjkQs"  # the video id whose transcript will be loaded by the API
try:
    ytt = YouTubeTranscriptApi()
    transcript_list = ytt.fetch(video_id, languages=["en"])

    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

[Submit subtitle corrections at criblate.com]
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user

In [11]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Submit subtitle corrections at criblate.com]\nImagine you happen across a short movie script that', start=1.14, duration=2.836), FetchedTranscriptSnippet(text='describes a scene between a person and their AI assistant.', start=3.976, duration=3.164), FetchedTranscriptSnippet(text="The script has what the person asks the AI, but the AI's response has been torn off.", start=7.48, duration=5.58), FetchedTranscriptSnippet(text='Suppose you also have this powerful magical machine that can take', start=13.06, duration=3.92), FetchedTranscriptSnippet(text='any text and provide a sensible prediction of what word comes next.', start=16.98, duration=3.98), FetchedTranscriptSnippet(text='You could then finish the script by feeding in what you have to the machine,', start=21.5, duration=4.006), FetchedTranscriptSnippet(text="seeing what it would predict to start the AI's answer,", start=25.506, duration=2.862), FetchedTranscriptSnippet(te

INDEXING USING TEXT SPLITTERS

In [12]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript]) 

In [13]:
len(chunks)

11

Embedding Generation (converting strings,audios,videos into vectors) and Storing in Vector Store

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [15]:

# 1. Use a fast, lightweight local embedding model (free, no API key required)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Create FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3109.44it/s]


In [16]:
vector_store.index_to_docstore_id

{0: '3d3f0ac4-8f41-4e83-bc1b-ccf96cc118da',
 1: '0a4e554f-687d-4f1d-b64c-05f4079a3afd',
 2: '69aa974b-fb32-4ff2-8a06-dcf0f6ff3df1',
 3: 'be60738a-f366-492b-8255-b7dd67407b59',
 4: '74a95772-217f-4f3a-9ee3-a8f9779110f1',
 5: '1639509d-cd7d-4091-90b8-fb478ae88bbd',
 6: '1bc2931c-20c9-4a1b-b43b-c7b358f9c9d7',
 7: 'c498c81a-aca3-446b-92a5-6094def44a47',
 8: '42396f6e-8707-4ec6-a087-78e7f440f6d2',
 9: '15048fbf-3444-429d-839d-b725103a83b5',
 10: '1e79b36e-3848-4568-8a91-70d5c1f76a04'}

In [17]:
vector_store.get_by_ids(['7d3a22c7-b9eb-40e5-9148-b10f391117bd'])

[]

RETRIEVER

In [18]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [19]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001EE389F4AA0>, search_kwargs={'k': 3})

In [20]:
retriever.invoke("What is LLM?")

[Document(id='42396f6e-8707-4ec6-a087-78e7f440f6d2', metadata={}, page_content="typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of this data repeatedly flows through many different iterations of these two fundamental operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's im

AUGMENTATION(merging relevant documents and user's query)

In [21]:
#  Use a valid model:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.7)
# or:
# llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.7)

In [22]:
prompt_template = PromptTemplate(
    template=""" 
    You are a helpful assistant. 
    Answer only from the provided transcript context.
    If you dont know the answer, say 'I don't know'.

    {context}
    Question:{question}
    """,
    input_variables=['context', 'question']
)

In [23]:
question="is the topic of neural networks discussed in the video?,if yes,explain in detail"
retrieved_docs=retriever.invoke(question)

In [24]:
retrieved_docs

[Document(id='42396f6e-8707-4ec6-a087-78e7f440f6d2', metadata={}, page_content="typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of this data repeatedly flows through many different iterations of these two fundamental operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's im

In [25]:
context_text="\n\n".join([doc.page_content for doc in retrieved_docs])
context_text

"typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of this data repeatedly flows through many different iterations of these two fundamental operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phenomenon\n\nI

In [26]:
final_prompt = prompt_template.invoke({"context": context_text, "question": question})
response = llm.invoke(final_prompt)
print(response.content)

Yes.  
The transcript explicitly mentions neural‑network components of a large language model. It says the model “typically also include a second type of operation known as a feed‑forward neural network.” This feed‑forward network is part of the core architecture that, together with the other fundamental operation (the attention‑based transformer layers), processes the sequence of numbers (embeddings) that represent the input text. The feed‑forward network gives the model additional capacity to store and manipulate patterns learned during training. As the data flows through many iterations of these two operations, the representations are enriched until a final function produces a probability distribution over possible next words. Thus, the discussion of neural networks—specifically the feed‑forward neural network component—is a key part of the explanation.


In [27]:
final_prompt

StringPromptValue(text=" \n    You are a helpful assistant. \n    Answer only from the provided transcript context.\n    If you dont know the answer, say 'I don't know'.\n\n    typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of this data repeatedly flows through many different iterations of these two fundamental operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possib

Generation

In [28]:
answer=llm.invoke(final_prompt)
print(answer.content)

Yes.  
The transcript explains that a large language model is built from two core operations: a **self‑attention mechanism** and a **feed‑forward neural network**. The feed‑forward part is a small neural network that sits inside each transformer block and gives the model extra capacity to remember patterns learned during training. The text also describes how the model repeatedly passes data through many iterations of these operations, gradually enriching the internal vectors until a final function produces a probability distribution over the next word. This description is a direct discussion of neural‑network‑based architecture and how it is trained and used in a chatbot.


currently it has different invokes for retriever,llm and answer

To Fix this we use chains that runs all invokes parrallely

In [29]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda,RunnableSequence
from langchain_core.output_parsers import StrOutputParser

In [38]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

parallel_chain = RunnableParallel({
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
})

# main_chain = parallel_chain | prompt_template | llm | parser

# # Invoke the entire chain in one call
# response = main_chain.invoke("is the topic of neural networks discussed in the video?, if yes, explain in detail")
# print(response)

In [39]:
parallel_chain=RunnableParallel({
    'context':retriever|RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [40]:
parallel_chain.invoke(question)

{'context': "typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to store more patterns about language learned during training. All of this data repeatedly flows through many different iterations of these two fundamental operations, and as it does so, the hope is that each list of numbers is enriched to encode whatever information might be needed to make an accurate prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phe

In [41]:
parser=StrOutputParser()

In [42]:
main_chain=parallel_chain|prompt_template|llm|parser

In [43]:
main_chain.invoke("Summarize this video")

'The video is an introductory look at transformer models, with a focus on the attention mechanism and the other steps that make up the architecture. The speaker explains that the behavior of large language models is an emergent phenomenon from billions of tuned parameters, which makes it difficult to predict exactly why a particular output is produced. They point out how LLMs can generate fluent and useful completions, and invite viewers to a deeper deep‑learning series that visualizes and motivates the details of attention. The speaker also notes that a more informal talk on the same topic was posted on a second channel for the company TNG in Munich, and mentions a personal preference for casual talks over produced videos, leaving the choice to the audience.'